In [33]:
%pip -q install pandas numpy openai tqdm

You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [34]:
from openai import OpenAI

client = OpenAI()  # uses OPENAI_API_KEY from env

models = client.models.list()
for m in models.data:
    print(m.id)


gpt-4-0613
gpt-4
gpt-3.5-turbo
gpt-5.2-codex
gpt-4o-mini-tts-2025-12-15
gpt-realtime-mini-2025-12-15
gpt-audio-mini-2025-12-15
chatgpt-image-latest
davinci-002
babbage-002
gpt-3.5-turbo-instruct
gpt-3.5-turbo-instruct-0914
dall-e-3
dall-e-2
gpt-4-1106-preview
gpt-3.5-turbo-1106
tts-1-hd
tts-1-1106
tts-1-hd-1106
text-embedding-3-small
text-embedding-3-large
gpt-4-0125-preview
gpt-4-turbo-preview
gpt-3.5-turbo-0125
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4o
gpt-4o-2024-05-13
gpt-4o-mini-2024-07-18
gpt-4o-mini
gpt-4o-2024-08-06
chatgpt-4o-latest
gpt-4o-audio-preview
gpt-4o-realtime-preview
omni-moderation-latest
omni-moderation-2024-09-26
gpt-4o-realtime-preview-2024-12-17
gpt-4o-audio-preview-2024-12-17
gpt-4o-mini-realtime-preview-2024-12-17
gpt-4o-mini-audio-preview-2024-12-17
o1-2024-12-17
o1
gpt-4o-mini-realtime-preview
gpt-4o-mini-audio-preview
o3-mini
o3-mini-2025-01-31
gpt-4o-2024-11-20
gpt-4o-search-preview-2025-03-11
gpt-4o-search-preview
gpt-4o-mini-search-preview-2025-03-11
gpt

In [35]:
import os
import re
import time
import pandas as pd
import numpy as np
from tqdm import tqdm

# ---------- Choose ONE provider ----------

# Option A) OpenAI (recommended if you have OPENAI_API_KEY set)
from openai import OpenAI
client = OpenAI()  # uses env var OPENAI_API_KEY

# Option B) HuggingFace Router / other OpenAI-compatible endpoint:
# from openai import OpenAI
# client = OpenAI(
#     base_url="https://router.huggingface.co/v1",
#     api_key=os.environ["HF_TOKEN"],  # set HF_TOKEN in env
# )

MODEL = "gpt-5.2"     # or "gpt-5.2-mini" if you want cheaper/faster
TEMPERATURE = 0
MAX_COMPLETION_TOKENS = 40


In [36]:
CSV_PATH = "../data/Ambivalent_posts_scoreVar_75_100_with_top10_comments_d0.csv"
df = pd.read_csv(CSV_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head(2)


Rows: 160
Columns: 24


,post_id,subreddit,title,author,score,num_comments_listed,op_replied,op_reply_count,created_utc,permalink,...,TC1,TC2,TC3,TC4,TC5,TC6,TC7,TC8,TC9,TC10
0,1bautv8,meToo,Not sure if this counts as SA,annoyingpea,4,2,False,0,1710025132,/r/meToo/comments/1bautv8/not_sure_if_this_cou...,...,"comment score : 5 | Comment : yes, this is ass...",comment score : 1 | Comment : I’m so sorry thi...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,w2py1z,meToo,was this sa?,babythecatto,6,13,False,0,1658228419,/r/meToo/comments/w2py1z/was_this_sa/,...,"comment score : 9 | Comment : This is abuse, m...","comment score : 3 | Comment : No, it is not no...",comment score : 2 | Comment : u/babythecatto w...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
TC_COLS = [f"TC{i}" for i in range(1, 11)]

def get_story_text(row: pd.Series) -> str:
    # Prefer combined_text, else body, else title
    for col in ["combined_text", "body", "title"]:
        if col in row and pd.notna(row[col]) and str(row[col]).strip():
            return str(row[col]).strip()
    return ""

def get_comments_list(row: pd.Series):
    comments = []
    for col in TC_COLS:
        if col in row and pd.notna(row[col]):
            txt = str(row[col]).strip()
            if txt:
                comments.append(txt)
    return comments


In [38]:
PROMPT_TEMPLATE = """

I will give you 
(1) a personal story describing sexual violence and 
(2) {n} responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank the responses by overall usefulness and reliability based only on the text provided. Do not assume facts that are not stated.

TIES ARE ALLOWED: Example (5 comments): Ranking: 2,1,3,3,5
This means comment [1] is ranked 2nd, comment [2] is ranked 1st, comments [3] and [4] are tied for 3rd, and comment [5] is ranked 5th.

IMPORTANT OUTPUT FORMAT:
Return EXACTLY one line in this format:
Ranking: <your ranking>
No extra text.

STORY:
{story}

RESPONSES:
{responses}
"""

def build_prompt(story: str, comments: list[str]) -> str:
    responses_block = "\n".join([f"{i}. {c}" for i, c in enumerate(comments, start=1)])
    return PROMPT_TEMPLATE.format(
        n=len(comments),
        story=story,
        responses=responses_block
    )


In [39]:
def call_ranker(prompt: str) -> str:
    last_err = None
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a careful evaluator of supportive responses. Follow instructions exactly."},
                    {"role": "user", "content": prompt},
                ],
                temperature=TEMPERATURE,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            last_err = e
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"LLM call failed after retries: {last_err}")


In [40]:
RANKING_RE = re.compile(r"^Ranking:\s*(.+)\s*$")

def parse_ranking_line(line: str, n: int) -> dict:
    """
    Returns:
      {
        "raw": original line,
        "ranking": extracted ranking string,
        "valid": bool,
        "error": str or None,
        "order_groups": list[list[int]]  # e.g., [[3,1],[2],[5,4]]
      }
    """
    out = {"raw": line, "ranking": None, "valid": False, "error": None, "order_groups": None}

    m = RANKING_RE.match(line.strip())
    if not m:
        out["error"] = "Missing/invalid 'Ranking:' prefix"
        return out

    ranking = m.group(1).strip()
    out["ranking"] = ranking

    # Split by commas into groups; each group may contain ties "="
    parts = [p.strip() for p in ranking.split(",") if p.strip()]
    groups = []
    seen = []

    for p in parts:
        tied = [x.strip() for x in p.split("=") if x.strip()]
        nums = []
        for t in tied:
            if not t.isdigit():
                out["error"] = f"Non-integer token: '{t}'"
                return out
            nums.append(int(t))
        groups.append(nums)
        seen.extend(nums)

    # Validate coverage and uniqueness
    expected = list(range(1, n + 1))
    if sorted(seen) != expected:
        out["error"] = f"Ranking must include each index exactly once from 1..{n}. Got: {sorted(seen)}"
        out["order_groups"] = groups
        return out

    out["valid"] = True
    out["order_groups"] = groups
    return out


In [41]:
# Pick any row index to test
test_i = 0

row = df.iloc[test_i]
story = get_story_text(row)
comments = get_comments_list(row)

print("post_id:", row.get("post_id"))
print("Num comments found:", len(comments))

prompt = build_prompt(story, comments)
print(prompt[:800], "\n...\n")

res = call_ranker(prompt)
print("MODEL OUTPUT:", res)

parsed = parse_ranking_line(res, n=len(comments))
parsed


post_id: 1bautv8
Num comments found: 2


I will give you 
(1) a personal story describing sexual violence and 
(2) 2 responses offering support or guidance. 
Your job is to evaluate the responses as replies to the storyteller.

Task: Rank the responses by overall usefulness and reliability based only on the text provided. Do not assume facts that are not stated.

TIES ARE ALLOWED: Example (5 comments): Ranking: 2,1,3,3,5
This means comment [1] is ranked 2nd, comment [2] is ranked 1st, comments [3] and [4] are tied for 3rd, and comment [5] is ranked 5th.

IMPORTANT OUTPUT FORMAT:
Return EXACTLY one line in this format:
Ranking: <your ranking>
No extra text.

STORY:
1bautv8 meToo Not sure if this counts as SA annoyingpea /r/meToo/comments/1bautv8/not_sure_if_this_counts_as_sa/ So yesterday I was at a party and I’ve definitely had 
...

MODEL OUTPUT: Ranking: 1,2


{'raw': 'Ranking: 1,2',
 'ranking': '1,2',
 'valid': True,
 'error': None,
 'order_groups': [[1], [2]]}

In [42]:
START_ROW = 0
END_ROW = len(df)          # you can set smaller for a first run
SLEEP_BETWEEN = 0.3        # pacing to reduce rate limits

results = []

for idx in tqdm(range(START_ROW, END_ROW)):
    row = df.iloc[idx]
    post_id = row.get("post_id", idx)

    story = get_story_text(row)
    comments = get_comments_list(row)
    n = len(comments)

    # Only skip if story is missing (optional)
    if not story:
        results.append({
            "row_index": idx,
            "post_id": post_id,
            "n_comments_used": n,
            "ranking_raw": None,
            "ranking_extracted": None,
            "ranking_valid": False,
            "ranking_error": "Skipped (missing story text)"
        })
        continue

    # If you're 100% sure n>=2 always, no need for any check here
    prompt = build_prompt(story, comments)

    try:
        raw = call_ranker(prompt)
        parsed = parse_ranking_line(raw, n=n)

        results.append({
            "row_index": idx,
            "post_id": post_id,
            "n_comments_used": n,
            "ranking_raw": raw,
            "ranking_extracted": parsed["ranking"],
            "ranking_valid": parsed["valid"],
            "ranking_error": parsed["error"],
        })

    except Exception as e:
        results.append({
            "row_index": idx,
            "post_id": post_id,
            "n_comments_used": n,
            "ranking_raw": None,
            "ranking_extracted": None,
            "ranking_valid": False,
            "ranking_error": str(e),
        })

    time.sleep(SLEEP_BETWEEN)

rank_df = pd.DataFrame(results)
rank_df.head(10)


100%|██████████| 160/160 [02:45<00:00,  1.03s/it]


,row_index,post_id,n_comments_used,ranking_raw,ranking_extracted,ranking_valid,ranking_error
0,0,1bautv8,2,"Ranking: 1,2","1,2",True,None
1,1,w2py1z,3,"Ranking: 1,3,2","1,3,2",True,None
2,2,1oecpi7,4,"Ranking: 2,4,1,3","2,4,1,3",True,None
3,3,1ochtx0,4,"Ranking: 2,1,3,4","2,1,3,4",True,None
4,4,1o7bk17,10,"Ranking: 6,8,3,1,2,5,4,10,7,9","6,8,3,1,2,5,4,10,7,9",True,None
5,5,1nzp70l,2,"Ranking: 1,2","1,2",True,None
6,6,1np7pss,2,"Ranking: 1,2","1,2",True,None
7,7,1jxruco,5,"Ranking: 4,3,2,1,5","4,3,2,1,5",True,None
8,8,1iv26kt,4,"Ranking: 1,2,4,3","1,2,4,3",True,None
9,9,1i3wqf5,9,"Ranking: 2,9,3,6,8,1,5,4,7","2,9,3,6,8,1,5,4,7",True,None


In [43]:
print("df columns sample:", df.columns.tolist()[:40])


df columns sample: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10']


In [44]:
print("df_out exists?", "df_out" in globals())
if "df_out" in globals():
    print("df_out shape:", df_out.shape)
    print("df_out columns contain post_id?", "post_id" in df_out.columns)
    print("df_out columns sample:", df_out.columns.tolist()[:40])

print("\nrank_df exists?", "rank_df" in globals())
if "rank_df" in globals():
    print("rank_df shape:", rank_df.shape)
    print("rank_df columns:", rank_df.columns.tolist())


df_out exists? True
df_out shape: (194, 25)
df_out columns contain post_id? True
df_out columns sample: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index']

rank_df exists? True
rank_df shape: (160, 7)
rank_df columns: ['row_index', 'post_id', 'n_comments_used', 'ranking_raw', 'ranking_extracted', 'ranking_valid', 'ranking_error']


In [45]:
# 1) Make sure df has a stable row_index to join on
df_out = df.copy()
df_out["row_index"] = range(len(df_out))

# 2) Merge ranking outputs back into the full df
# rank_df already has: row_index, post_id, n_comments_used, ranking_raw, ranking_extracted, ranking_valid, ranking_error
df_merged = df_out.merge(
    rank_df,
    on=["row_index", "post_id"],   # safest join keys
    how="left"
)

# 3) (Optional) arrange columns: original columns first, then ranking columns
orig_cols = [
    "post_id","subreddit","title","author","score","num_comments_listed","op_replied",
    "op_reply_count","created_utc","permalink","body","combined_text",
    "matched_ambivalent_phrases","matched_keywords",
    "TC1","TC2","TC3","TC4","TC5","TC6","TC7","TC8","TC9","TC10"
]

rank_cols = ["row_index","n_comments_used","ranking_raw","ranking_extracted","ranking_valid","ranking_error"]

# keep any extra original cols (if they exist) without breaking
extras = [c for c in df_merged.columns if c not in orig_cols + rank_cols]
df_merged = df_merged[orig_cols + extras + rank_cols]

# 4) Save
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_rankings-75-100.csv"
df_merged.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, "| shape:", df_merged.shape)


Saved: ../data/Ambivalent_posts_with_top10_comments_with_rankings-75-100.csv | shape: (160, 30)


In [47]:
pip install scipy

     |████████████████████████████████| 38.6 MB 59.2 MB/s            ██████▎                | 18.4 MB 8.2 MB/s eta 0:00:03
You should consider upgrading via the '/home/ec2-user/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [48]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50.csv
Shape: (195, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 195 / 195
Mean Kendall Tau-b (IRA): 0.1967844147521091
Mean Spearman Rho: 0.19833756375580863

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_25_50_with_IRA.csv


In [49]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75.csv
Shape: (194, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 194 / 194
Mean Kendall Tau-b (IRA): 0.2660290288746958
Mean Spearman Rho: 0.290542857411822

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_50_75_with_IRA.csv


In [50]:
import re
import numpy as np
import pandas as pd
from scipy.stats import kendalltau, spearmanr

# =========================
# 1) Load input
# =========================
IN_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100.csv"

# If your file is tab-separated, switch to: pd.read_csv(IN_PATH, sep="\t")
df = pd.read_csv(IN_PATH)

print("Loaded:", IN_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# =========================
# 2) Helpers
# =========================
def extract_ints(s: str):
    """Extract integers from a string like '1,1,3' or 'Ranking: 1, 1, 3' or '[1,1,3]'."""
    return [int(x) for x in re.findall(r"-?\d+", str(s))]

def parse_rankvec(s, n: int):
    """
    Parse a per-comment rank vector from a cell like:
      "1,1,3" or "1, 1, 3" or "[1,1,3]" or "Ranking: 1, 1, 3"
    Returns: list[int] of length n
    """
    if pd.isna(s):
        raise ValueError("Ranking is NaN")

    nums = extract_ints(s)

    if len(nums) != n:
        raise ValueError(f"Expected {n} ranks, got {len(nums)} -> {nums}")

    if any(r <= 0 for r in nums):
        raise ValueError(f"Ranks must be positive integers -> {nums}")

    return nums

def infer_n_from_rankings(row):
    """
    Prefer n_comments_used if present/valid, otherwise infer from Human_Rankings length,
    otherwise infer from LLM_Rankings length.
    """
    n = row.get("n_comments_used", np.nan)

    if not pd.isna(n):
        try:
            n = int(n)
            if n > 0:
                return n
        except Exception:
            pass

    # fallback: infer from human ranking length
    h = row.get("Human_Rankings", None)
    if h is not None and not pd.isna(h):
        hn = len(extract_ints(h))
        if hn > 0:
            return hn

    # fallback: infer from llm ranking length
    m = row.get("LLM_Rankings", None)
    if m is not None and not pd.isna(m):
        mn = len(extract_ints(m))
        if mn > 0:
            return mn

    raise ValueError("Could not infer n (n_comments_used missing/invalid and rankings empty).")

def compute_ira_row(row):
    n = infer_n_from_rankings(row)

    human = parse_rankvec(row["Human_Rankings"], n)
    llm   = parse_rankvec(row["LLM_Rankings"], n)

    # Kendall tau-b handles ties properly
    tau, tau_p = kendalltau(human, llm, variant="b")

    # Spearman rho (also supports ties)
    rho, rho_p = spearmanr(human, llm)

    return pd.Series({
        "n_used_for_ira": n,
        "ira_kendall_tau_b": tau,
        "ira_kendall_p": tau_p,
        "ira_spearman_rho": rho,
        "ira_spearman_p": rho_p,
        "ira_valid": True,
        "ira_error": ""
    })

# =========================
# 3) Compute IRA per row
# =========================
required = {"Human_Rankings", "LLM_Rankings"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

ira_out = []
for _, row in df.iterrows():
    try:
        ira_out.append(compute_ira_row(row))
    except Exception as e:
        ira_out.append(pd.Series({
            "n_used_for_ira": np.nan,
            "ira_kendall_tau_b": np.nan,
            "ira_kendall_p": np.nan,
            "ira_spearman_rho": np.nan,
            "ira_spearman_p": np.nan,
            "ira_valid": False,
            "ira_error": str(e)
        }))

ira_df = pd.DataFrame(ira_out)
df_with_ira = pd.concat([df.reset_index(drop=True), ira_df], axis=1)

# =========================
# 4) Summary (overall IRA)
# =========================
valid = df_with_ira["ira_valid"] == True
print("\nValid rows:", int(valid.sum()), "/", len(df_with_ira))
print("Mean Kendall Tau-b (IRA):", df_with_ira.loc[valid, "ira_kendall_tau_b"].mean())
print("Mean Spearman Rho:", df_with_ira.loc[valid, "ira_spearman_rho"].mean())

# Show a few failures (if any)
if (~valid).any():
    print("\nSample invalid rows:")
    cols_to_show = [c for c in ["post_id", "n_comments_used", "Human_Rankings", "LLM_Rankings", "ira_error"] if c in df_with_ira.columns]
    print(df_with_ira.loc[~valid, cols_to_show].head(10).to_string(index=False))

# =========================
# 5) Save
# =========================
OUT_PATH = "../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100_with_IRA.csv"
df_with_ira.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


Loaded: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100.csv
Shape: (160, 31)
Columns: ['post_id', 'subreddit', 'title', 'author', 'score', 'num_comments_listed', 'op_replied', 'op_reply_count', 'created_utc', 'permalink', 'body', 'combined_text', 'matched_ambivalent_phrases', 'matched_keywords', 'TC1', 'TC2', 'TC3', 'TC4', 'TC5', 'TC6', 'TC7', 'TC8', 'TC9', 'TC10', 'row_index', 'n_comments_used', 'ranking_raw', 'ranking_valid', 'ranking_error', 'Human_Rankings', 'LLM_Rankings']

Valid rows: 159 / 160
Mean Kendall Tau-b (IRA): 0.37701113492725724
Mean Spearman Rho: 0.4281988516911145

Sample invalid rows:
post_id  n_comments_used Human_Rankings LLM_Rankings                               ira_error
11t1ste                3          1,2,2      2,3,3,1 Expected 3 ranks, got 4 -> [2, 3, 3, 1]

Saved: ../data/Ambivalent_posts_with_top10_comments_with_Human_Rankings_75_100_with_IRA.csv
